# Prompt and in-context learning experiments with Cosmos

**Time:** 2–3 hours (Cosmos GPU)  
**Prerequisites:** notebooks 04–05, configured `COSMOS`, and one or more captured images.

## Learning objectives

- Run controlled prompt experiments on a frozen image set
- Measure JSON validity, semantic correctness, latency, and consistency
- Use image/answer examples for classification or orientation—not invented robot coordinates
- Choose a prompt from evidence rather than a memorable single result

> The production cube pipeline is hybrid: calibrated image geometry measures pose and Cosmos confirms it. This notebook must not turn model-only numeric pose guesses into grasp commands.

In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'perceive.py').is_file(): return p
    raise FileNotFoundError('Repository root not found')
ROOT = find_root(); SIM = ROOT / 'fanuc_er4ia_sim'; RUNS = SIM / 'runs'
COSMOS = os.environ.get('COSMOS')
images = sorted(RUNS.glob('*/world_camera.png')) if RUNS.exists() else []
print('COSMOS:', COSMOS or 'NOT SET')
print('Frozen images:', len(images))
for p in images[:10]: print(' ', p.relative_to(SIM))

## 1. Design the experiment first

Keep the image set, model snapshot, decoding settings, and scoring rules fixed. Change only the prompt or example set. Decide success before inference. For a simple cube-colour task, score: valid JSON, allowed labels only, expected item count, and agreement across repeated runs.

Good prompt variants differ for a reason: concise baseline, explicit exclusions, and explicit schema. Avoid changing ten phrases at once.

In [ ]:
PROMPTS = {
 'baseline': '''List the coloured cubes in this image. Reply with JSON only: {"objects": [{"colour": "red|yellow|blue"}]}''',
 'explicit_exclusions': '''Find only the six small plastic cubes. Ignore the robot, bins, caps, tubes, paper, hoses, and background. Reply with JSON only: {"objects": [{"colour": "red|yellow|blue"}]}''',
 'schema_and_count': '''Find every small plastic cube. There should normally be two red, two yellow, and two blue, but report only what is visible. Ignore all non-cubes. Reply with exactly one JSON object and no prose: {"objects": [{"colour": "red|yellow|blue"}], "visible_count": 0}''',
}
for name, text in PROMPTS.items(): print(name, '->', text[:100], '...')

## 2. Load Cosmos once, then reuse it

Model startup is expensive. Run the next cell only in the Cosmos-capable kernel/environment. If this tutorial kernel is not the Cosmos interpreter, use the subprocess harness later instead.

In [ ]:
# Optional direct mode. It may take several minutes and substantial GPU memory.
LOAD_MODEL = False  # Opt in only from the configured Cosmos environment.
model = processor = None
try:
    if not LOAD_MODEL: raise RuntimeError('Direct model loading not requested')
    import sys
    sys.path.insert(0, str(SIM))
    from perceive import load_model, ask, parse_json, canonicalise
    model, processor = load_model()
    print('Cosmos loaded')
except Exception as exc:
    print('Direct mode unavailable:', type(exc).__name__, exc)
    print('Use the Cosmos interpreter or the production CLI commands below.')

In [ ]:
def score_reply(raw):
    try: data = json.loads(raw)
    except json.JSONDecodeError: return {'valid_json': False, 'allowed_labels': False, 'count': None}
    objects = data.get('objects', []) if isinstance(data, dict) else []
    labels = [o.get('colour') for o in objects if isinstance(o, dict)]
    return {'valid_json': True, 'allowed_labels': all(x in {'red','yellow','blue'} for x in labels), 'count': len(labels)}

results = []
if model is not None and images:
    for image in images[:3]:
        for prompt_name, prompt in PROMPTS.items():
            start = time.perf_counter()
            raw = ask(model, processor, image, prompt, max_new_tokens=500)
            row = {'image': str(image), 'prompt': prompt_name, 'latency_s': time.perf_counter()-start, 'raw': raw}
            row.update(score_reply(raw)); results.append(row)
            print(prompt_name, image.parent.name, score_reply(raw), f"{row['latency_s']:.1f}s")
else: print('Skipped: direct Cosmos model or images are unavailable.')

## 3. Exercise the production prompt

The safest end-to-end comparison is to copy the same captured image into separate run directories and invoke the supported pipeline entry point. Each output directory preserves raw text and parsed JSON.

```bash
"$COSMOS" fanuc_er4ia_sim/perceive.py --run fanuc_er4ia_sim/runs/prompt_baseline --image fanuc_er4ia_sim/runs/stack_01/world_camera.png --stack-cubes
```

Do not overwrite a run whose artifacts you still need. Record the model snapshot and prompt source revision with every comparison.

## 4. In-context learning

`perceive.ask(..., examples=[(image_path, answer), ...])` adds image/answer demonstrations before the new query. Good examples are representative, correctly labelled, and cover the decision boundary. For example, demonstrate both `narrow_end_up` and `wide_end_up` caps.

Bad uses include putting hidden simulator coordinates in the answer, teaching from evaluation truth that will not exist later, or using near-duplicate examples from the test image.

In [ ]:
EXAMPLE_DIR = SIM / 'assets' / 'perception_examples'
example_files = sorted(EXAMPLE_DIR.glob('*')) if EXAMPLE_DIR.exists() else []
print('Available example assets:')
for p in example_files: print(' ', p.name)

# Fill these only with reviewed crop images and exact expected JSON answers.
icl_examples = [
    # (EXAMPLE_DIR / 'cap_narrow.png', '{"orientation": "narrow_end_up"}'),
    # (EXAMPLE_DIR / 'cap_wide.png',   '{"orientation": "wide_end_up"}'),
]
assert all(path.exists() for path, _ in icl_examples)

In [ ]:
ICL_PROMPT = '''Classify the cap nearest the centre. Reply with JSON only: {"orientation": "narrow_end_up|wide_end_up"}'''
TARGET_CROP = None  # Example: RUNS / 'stack_01' / 'cap_0_crop.png'
if model is not None and TARGET_CROP and Path(TARGET_CROP).exists() and icl_examples:
    zero_shot = ask(model, processor, TARGET_CROP, ICL_PROMPT, max_new_tokens=120)
    two_shot = ask(model, processor, TARGET_CROP, ICL_PROMPT, max_new_tokens=120, examples=icl_examples)
    print('zero-shot:', zero_shot)
    print('two-shot:', two_shot)
else:
    print('Add two reviewed example images and a held-out TARGET_CROP to run this comparison.')

## 5. Analyze consistency, not just best case

Repeat each condition at least three times if decoding is nondeterministic. A prompt that succeeds once and emits malformed JSON twice is not robust. Keep raw replies: parse failures are data.

In [ ]:
for name in PROMPTS:
    rows = [r for r in results if r['prompt'] == name]
    if not rows: continue
    print(name, {
        'n': len(rows),
        'json_rate': sum(r['valid_json'] for r in rows)/len(rows),
        'label_rate': sum(r['allowed_labels'] for r in rows)/len(rows),
        'mean_latency_s': sum(r['latency_s'] for r in rows)/len(rows),
        'counts': [r['count'] for r in rows],
    })

## Team exercise and checkpoint

Create a small frozen test set with at least one easy, one rotated, and one visually difficult scene. Compare the three prompts plus zero-shot versus two-shot cap orientation. Write a short decision note containing the test images, prompt text, model snapshot, number of repeats, metrics, failures, and chosen prompt.

You are finished when a teammate can rerun the comparison and reach the same conclusion. A prompt is not promoted merely because its best output looks persuasive.